## MODHE

In [4]:
from sage.all import *

# --- timing for the thesis: wall clock per operation, seconds ---
from time import perf_counter

timings = []


def measure(label, fn, *args, **kwargs):
    t0 = perf_counter()
    out = fn(*args, **kwargs)
    seconds = perf_counter() - t0
    timings.append((label, seconds))
    print(f"[time] {label}: {seconds:.1f} s")
    return out


In [5]:
load("../MODHE/constants.sage")
load("../MODHE/auxiliary_functions.sage")
load("../MODHE/KeyGen.sage")
load("../MODHE/Encrypt-Decrypt.sage")
load("../MODHE/HMult.sage")
load("../MODHE/Relin.sage")

In [6]:
print("\nGenerating Keys (this may take a moment)...")
key = measure("keygen", public_private_keygen)
evk_key = measure("relin_keygen", relin_keygen, key)


Generating Keys (this may take a moment)...
private-public keypair generated
[time] keygen: 40.0 s
[time] relin_keygen: 1106.6 s


In [7]:
m_plain1 = 0.4
m_plain2 = 0.6

ct1 = measure("encrypt_1", encryption, m_plain1, key)
ct2 = measure("encrypt_2", encryption, m_plain2, key)

print(f"\nEncrypted m_plain1. Decryption check: {measure("decrypt_1", decryption, ct1, key, 0, 0)}")
print(f"Encrypted m_plain2. Decryption check: {measure("decrypt_2", decryption, ct2, key, 0, 0)}")

m= 0.400000000000000
[time] encrypt_1: 72.0 s
m= 0.600000000000000
[time] encrypt_2: 71.5 s
decryption:
[time] decrypt_1: 0.1 s

Encrypted m_plain1. Decryption check: 0.39999999998649216
decryption:
[time] decrypt_2: 0.1 s
Encrypted m_plain2. Decryption check: 0.5999999999863266


In [8]:
# Step A: Homomorphic Multiplication (Outputs extended 6-part ciphertext)
mult_result = measure("hmult", homomorphic_mult, ct1, ct2, L-1, key)

[time] hmult: 40.2 s


In [9]:
# Step B: Relinearization (Squashes back to 2-part ciphertext)
relin_result = measure("relin", relinearisation, mult_result, evk_key)

[time] relin: 5077.9 s


In [10]:
# Step C: Decrypt and Rescale (Drops from level L-1 to L-2)
final_HE_dec = measure("decrypt_after_rescale", decryption, relin_result, key, L-1, L-2)

decryption:
[time] decrypt_after_rescale: 4036.6 s


In [11]:
print("\n--- RESULTS ---")
print(f"Expected Result (0.4 * 0.6): {m_plain1 * m_plain2}")
print(f"HE Computed Result:          {final_HE_dec}")


--- RESULTS ---
Expected Result (0.4 * 0.6): 0.240000000000000
HE Computed Result:          0.23999999744481082


In [12]:
# --- write the timing table: one row per operation, parameters repeated ---
import csv
import os
from datetime import datetime

path = "sagemath_timing.csv"
params = [("prototype", "modhe_baseline"), ("L", L), ("MLWE", MLWE), ("POLDEG", POLDEG)]
new_file = not os.path.exists(path)
stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(path, "a", newline="") as fh:
    writer = csv.writer(fh)
    if new_file:
        writer.writerow([name for name, _ in params] + ["operation", "seconds", "run"])
    for label, seconds in timings:
        writer.writerow([value for _, value in params] + [label, f"{seconds:.2f}", stamp])

print("\n--- TIMING (one run, this machine) ---")
for label, seconds in timings:
    print(f"{label:26s} {seconds:8.1f} s")
print(f"written to {path}")



--- TIMING (one run, this machine) ---
keygen                         40.0 s
relin_keygen                 1106.6 s
encrypt_1                      72.0 s
encrypt_2                      71.5 s
decrypt_1                       0.1 s
decrypt_2                       0.1 s
hmult                          40.2 s
relin                        5077.9 s
decrypt_after_rescale        4036.6 s
written to sagemath_timing.csv
